### **Sieć neuronowa STL na deskryptorach**


Wykorzystana reprezentacja: **10-cechowe deskryptory 2D**

**Lista endpointów**:

1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. Half Life (Obach)
6. Clearance Hepatocyte (AstraZeneca)
7. CYP3A4 Inhibition (Veith)
8. VDss (Lombardo)
9. AMES Mutagenicity
10. hERG (Wang)

**Metryki jakości**: AUROC (klasyfikacja), RMSE(regresja), Accuracy, F1 (klasyfikacja), R^2(regresja), MAE(regresja)

In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_folder = "/content/drive/MyDrive/mldd_data/" #folder z zapisanymi zestawami danych aby umożliwić porównanie danych na dokładnie tych samych zestawach dla każdego modelu

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score


# DEFINICJA SIECI NEURONOWEJ

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))


class ADMETDescriptorFeaturizer:
    def __init__(self, y_column, smiles_col='Drug'):
        self.y_column = y_column
        self.smiles_col = smiles_col
        self.feature_names = [
            'MW', 'LogP', 'HBD', 'HBA', 'TPSA',
            'RotatableBonds', 'AromaticRings', 'HeavyAtoms',
            'MolMR', 'FractionCSP3'
        ]
        self.scaler = StandardScaler()

    def __call__(self, df, fit_scaler=True):
        features = []
        labels = []

        for i, row in df.iterrows():
            smiles = row[self.smiles_col]
            y = row[self.y_column]
            mol = Chem.MolFromSmiles(smiles)

            if mol:
                desc_vector = [
                    Descriptors.MolWt(mol),
                    Descriptors.MolLogP(mol),
                    Lipinski.NumHDonors(mol),
                    Lipinski.NumHAcceptors(mol),
                    Descriptors.TPSA(mol),
                    Lipinski.NumRotatableBonds(mol),
                    Lipinski.NumAromaticRings(mol),
                    Descriptors.HeavyAtomCount(mol),
                    Descriptors.MolMR(mol),
                    Lipinski.FractionCSP3(mol)
                ]
                features.append(desc_vector)
                labels.append(y)

        features_array = np.array(features)

        if fit_scaler:
            normalized_features = self.scaler.fit_transform(features_array)
        else:
            normalized_features = self.scaler.transform(features_array)

        return normalized_features, np.array(labels).reshape(-1, 1)



In [ ]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np):

    # Normalizacja
    print("Normalizacja danych...")
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train_np)
    y_test_scaled = scaler.transform(y_test_np)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_scaled).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = STL_NN_Regressor(input_dim=len(featurizer.feature_names)).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("\n--- START TRENINGU (Regresja) ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            mse_val = loss.item()
            print(f"  Epoka {epoch:3d} | Błąd MSE: {loss.item():.4f}")

# Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).cpu().numpy()

    preds = scaler.inverse_transform(preds_scaled)

    mse = mean_squared_error(y_test_np, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_np, preds)
    r2 = r2_score(y_test_np, preds)

    metrics = {
        "test_metrics": {
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }
    }

    return metrics


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
def train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_np).to(device) # Etykiety 0.0 lub 1.0
    X_test_t = torch.FloatTensor(X_test_np).to(device)

# Model i Trening
    model = STL_NN_Classifier(input_dim=len(featurizer.feature_names)).to(device)
    criterion = nn.BCELoss() # Binary Cross Entropy
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("\n--- START TRENINGU (Klasyfikacja) ---")
    model.train()
    for epoch in range(101):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Błąd BCE Loss: {loss.item():.4f}")

# Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        # Model zwraca prawdopodobieństwo (0 do 1)
        probs = model(X_test_t).cpu().numpy()
        # Klasy (0 lub 1) ustalamy progiem 0.5
    preds = (probs > 0.5).astype(int)


    auroc = roc_auc_score(y_test_np, probs)
    acc = accuracy_score(y_test_np, preds)

    print(f"Wynik końcowy AUROC (HIA): {auroc:.4f}")
    print(f"Dokładność (Accuracy): {acc*100:.2f}%")

    metrics = {
        "test_metrics": {
            "accuracy": accuracy_score(y_test_np, preds),
            "f1":       f1_score(y_test_np, preds),
            "auroc":    roc_auc_score(y_test_np, probs),
        }
    }

    return metrics

Endpoint 1: Caco-2 (Wang)

In [ ]:
train, test = load_split_pickle('Caco2_Wang')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Caco2_Wang", "metrics_NN_descriptors.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.6074
  Epoka  20 | Błąd MSE: 0.3871
  Epoka  40 | Błąd MSE: 0.2923
  Epoka  60 | Błąd MSE: 0.2423
  Epoka  80 | Błąd MSE: 0.2102

--- EWALUACJA ---

  RMSE     : 0.4763
  MAE      : 0.3369
  R²       : 0.6428



Endpoint 2:  Lipophilicity

In [ ]:
train, test = load_split_pickle('Lipophilicity_AstraZeneca')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Lipophilicity_AstraZeneca", "metrics_NN_descriptors.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1473
  Epoka  20 | Błąd MSE: 0.6904
  Epoka  40 | Błąd MSE: 0.6171
  Epoka  60 | Błąd MSE: 0.5710
  Epoka  80 | Błąd MSE: 0.5306

--- EWALUACJA ---

  RMSE     : 0.8958
  MAE      : 0.6804
  R²       : 0.4568



Endpoint 3: Solubility

In [ ]:
train, test = load_split_pickle('Solubility_AqSolDB')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Solubility_AqSolDB", "metrics_NN_descriptors.txt", task="regression")

[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:41] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not removing hydrogen atom without neighbors
[23:17:42] WARNING: not r

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.2876
  Epoka  20 | Błąd MSE: 0.3849
  Epoka  40 | Błąd MSE: 0.3337
  Epoka  60 | Błąd MSE: 0.3013
  Epoka  80 | Błąd MSE: 0.2890

--- EWALUACJA ---

  RMSE     : 1.2061
  MAE      : 0.8663
  R²       : 0.7319



Endpoint 4: HIA

In [ ]:
train, test = load_split_pickle('HIA_Hou')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "HIA_Hou", "metrics_NN_descriptors.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.5826
  Epoka  20 | Błąd BCE Loss: 0.2445
  Epoka  40 | Błąd BCE Loss: 0.1819
  Epoka  60 | Błąd BCE Loss: 0.1320
  Epoka  80 | Błąd BCE Loss: 0.1031
  Epoka 100 | Błąd BCE Loss: 0.0807

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.9300
Dokładność (Accuracy): 90.52%

  Accuracy : 0.9052
  F1       : 0.9453
  AUROC    : 0.9300



Endpoint 5: Half Life

In [ ]:
train, test = load_split_pickle('Half_Life_Obach')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Half_Life_Obach", "metrics_NN_descriptors.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.0847
  Epoka  20 | Błąd MSE: 0.9127
  Epoka  40 | Błąd MSE: 0.8097
  Epoka  60 | Błąd MSE: 0.7055
  Epoka  80 | Błąd MSE: 0.6168

--- EWALUACJA ---

  RMSE     : 98.7200
  MAE      : 27.7646
  R²       : 0.3206



Endpoint 6: Clearance Hepatocyte

In [ ]:
train, test = load_split_pickle('Clearance_Hepatocyte_AZ')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Clearance_Hepatocyte_AZ", "metrics_NN_descriptors.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1395
  Epoka  20 | Błąd MSE: 0.8565
  Epoka  40 | Błąd MSE: 0.8234
  Epoka  60 | Błąd MSE: 0.7551
  Epoka  80 | Błąd MSE: 0.7134

--- EWALUACJA ---

  RMSE     : 47.2170
  MAE      : 34.9098
  R²       : 0.1069



Endpoint 7: CYP3A4 Inhibition

In [ ]:
train, test = load_split_pickle('CYP3A4_Veith')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "CYP3A4_Veith", "metrics_NN_descriptors.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.6990
  Epoka  20 | Błąd BCE Loss: 0.5034
  Epoka  40 | Błąd BCE Loss: 0.4926
  Epoka  60 | Błąd BCE Loss: 0.4812
  Epoka  80 | Błąd BCE Loss: 0.4749
  Epoka 100 | Błąd BCE Loss: 0.4655

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8236
Dokładność (Accuracy): 74.53%

  Accuracy : 0.7453
  F1       : 0.6937
  AUROC    : 0.8236



Endpoint 8: VDss

In [ ]:
train, test = load_split_pickle('VDss_Lombardo')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "VDss_Lombardo", "metrics_NN_descriptors.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.2920
  Epoka  20 | Błąd MSE: 0.9529
  Epoka  40 | Błąd MSE: 0.8907
  Epoka  60 | Błąd MSE: 0.8202
  Epoka  80 | Błąd MSE: 0.7381

--- EWALUACJA ---

  RMSE     : 11.1298
  MAE      : 5.6490
  R²       : -1.6535



Endpoint 9: AMES Mutagenicity

In [ ]:
train, test = load_split_pickle('AMES')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "AMES", "metrics_NN_descriptors.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.7068
  Epoka  20 | Błąd BCE Loss: 0.5968
  Epoka  40 | Błąd BCE Loss: 0.5737
  Epoka  60 | Błąd BCE Loss: 0.5535
  Epoka  80 | Błąd BCE Loss: 0.5368
  Epoka 100 | Błąd BCE Loss: 0.5195

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8085
Dokładność (Accuracy): 73.21%

  Accuracy : 0.7321
  F1       : 0.7464
  AUROC    : 0.8085



Endpoint 10: hERG (Wang)

In [ ]:
train, test = load_split_pickle('hERG')

featurizer = ADMETDescriptorFeaturizer(y_column='Y')

X_train_np, y_train_np = featurizer(train, fit_scaler=True)
X_test_np, y_test_np = featurizer(test, fit_scaler=False)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "hERG", "metrics_NN_descriptors.txt")

[23:18:16] WARNING: not removing hydrogen atom without neighbors
[23:18:16] WARNING: not removing hydrogen atom without neighbors



--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.7766
  Epoka  20 | Błąd BCE Loss: 0.3754
  Epoka  40 | Błąd BCE Loss: 0.3195
  Epoka  60 | Błąd BCE Loss: 0.2779
  Epoka  80 | Błąd BCE Loss: 0.2481
  Epoka 100 | Błąd BCE Loss: 0.2073

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.7883
Dokładność (Accuracy): 83.21%

  Accuracy : 0.8321
  F1       : 0.8911
  AUROC    : 0.7883

